In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q peft

import pandas as pd
import torch
import numpy as np

from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

label2id = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
id2label = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

train["label"] = train["answer"].map(label2id)

train.head()

,id,prompt,A,B,C,D,E,answer,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,2
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,0


# Multiple-Choice Data Formatting
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.
- Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4

- What is the encoded numeric label for the row at index 150?

In [3]:
answer_150 = train.loc[150, "answer"]
encoded_150 = label2id[answer_150]

print("Answer at index 150:", answer_150)
print("Encoded label:", encoded_150)

Answer at index 150: C
Encoded label: 2


# Q2. Prompt-Option Formatting
- For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

- What is the exact character length of this formatted input string?

In [4]:
formatted_B = str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, "B"])

print(formatted_B)
print("Length:", len(formatted_B))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Length: 407


# Tokenization for Multiple-Choice Models
### Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.
- Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

- What is the value of the second dimension?
*


In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row = train.loc[0]

five_inputs = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoded = tokenizer(
    five_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoded["input_ids"].unsqueeze(0)

print(input_ids.shape)
print("Second dimension:", input_ids.shape[1])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([1, 5, 128])
Second dimension: 5


# Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [6]:
all_inputs = []

for i in range(16):
    row = train.loc[i]
    choices = ["A", "B", "C", "D", "E"]
    
    row_inputs = [
        str(row["prompt"]) + " [SEP] " + str(row[ch])
        for ch in choices
    ]
    
    all_inputs.extend(row_inputs)

encoded_batch = tokenizer(
    all_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids_batch = encoded_batch["input_ids"].view(16, 5, 128)

print(input_ids_batch.shape)

total_positions = 16 * 5 * 128
print("Total token positions:", total_positions)

torch.Size([16, 5, 128])
Total token positions: 10240


# Multiple-Choice Model Outputs
AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.
- Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

In [7]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

row = train.loc[0]

five_inputs = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoded = tokenizer(
    five_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

mcq_inputs = {
    "input_ids": encoded["input_ids"].unsqueeze(0),
    "attention_mask": encoded["attention_mask"].unsqueeze(0)
}

with torch.no_grad():
    outputs = model(**mcq_inputs)

print(outputs.logits.shape)
print("Number of logits:", outputs.logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])
Number of logits: 5


# Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?
*


In [8]:
correct_label = torch.tensor([train.loc[0, "label"]])

with torch.no_grad():
    outputs_with_loss = model(
        input_ids=mcq_inputs["input_ids"],
        attention_mask=mcq_inputs["attention_mask"],
        labels=correct_label
    )

loss = outputs_with_loss.loss

print(loss)
print("Loss tensor dimensions:", loss.dim())

tensor(1.6053)
Loss tensor dimensions: 0


# LoRA for Efficient Fine-Tuning
- LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.
- Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?
*


In [9]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel() for p in lora_model.parameters() if p.requires_grad
)

print("Trainable parameters:", trainable_params)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 295681


# Preparing Data for Hugging Face Trainer
- Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.
- Q8. Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?
*


In [10]:
small_df = train.iloc[:100].copy()

def tokenize_mcq_example(example):
    choices = ["A", "B", "C", "D", "E"]
    
    texts = [
        str(example["prompt"]) + " [SEP] " + str(example[ch])
        for ch in choices
    ]
    
    encoded = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )
    
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": int(example["label"])
    }

hf_dataset = Dataset.from_pandas(small_df)
tokenized_dataset = hf_dataset.map(tokenize_mcq_example)

first_item = tokenized_dataset[0]

print(np.array(first_item["input_ids"]).shape)
print("Number of tokenized choices:", len(first_item["input_ids"]))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

(5, 128)
Number of tokenized choices: 5


# Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.
- Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

- Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

- What is the final global_step reported by the Trainer?
*


In [11]:
from dataclasses import dataclass
from typing import Optional, Union
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy

MAX_LENGTH = 64

tiny_df = train.iloc[:32].copy()
tiny_dataset = Dataset.from_pandas(tiny_df)

def tokenize_tiny(example):
    choices = ["A", "B", "C", "D", "E"]
    
    texts = [
        str(example["prompt"]) + " [SEP] " + str(example[ch])
        for ch in choices
    ]
    
    encoded = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )
    
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": int(example["label"])
    }

tiny_tokenized = tiny_dataset.map(
    tokenize_tiny,
    remove_columns=tiny_dataset.column_names
)

@dataclass
class DataCollatorForMCQ:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    
    def __call__(self, features):
        labels = torch.tensor([f.pop("labels") for f in features], dtype=torch.long)
        
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened = []
        for feature in features:
            for i in range(num_choices):
                flattened.append({
                    "input_ids": feature["input_ids"][i],
                    "attention_mask": feature["attention_mask"][i]
                })
        
        batch = self.tokenizer.pad(
            flattened,
            padding=self.padding,
            return_tensors="pt"
        )
        
        batch = {
            k: v.view(batch_size, num_choices, -1)
            for k, v in batch.items()
        }
        
        batch["labels"] = labels
        return batch

data_collator = DataCollatorForMCQ(tokenizer)

base_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(base_model, lora_config)

training_args = TrainingArguments(
    output_dir="./tiny_lora_mcq",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=False,
    bf16=False,
    remove_unused_columns=False
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tiny_tokenized,
    data_collator=data_collator
)

train_result = trainer.train()

print("Global step:", trainer.state.global_step)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,1.574339
2,1.570907
3,1.655049
4,1.630148


Global step: 4


# Q10. Probability Assigned to Option E After Fine-Tuning
- Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

- What is the probability assigned to Option E?

Round your answer to 4 decimal places.
*


In [12]:
row = train.loc[0]

texts = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoded = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

device = trainer.model.device

input_ids = encoded["input_ids"].unsqueeze(0).to(device)
attention_mask = encoded["attention_mask"].unsqueeze(0).to(device)

trainer.model.eval()

with torch.no_grad():
    outputs = trainer.model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

logits = outputs.logits
probs = torch.softmax(logits, dim=1)

option_e_prob = probs[0, 4].item()

print("Logits:", logits)
print("Probabilities:", probs)
print("Option E probability:", round(option_e_prob, 4))

Logits: tensor([[0.3973, 0.3496, 0.3716, 0.4061, 0.4325]])
Probabilities: tensor([[0.2011, 0.1917, 0.1960, 0.2029, 0.2083]])
Option E probability: 0.2083
